# SCIDOCS Dense-Retrieval Baseline (no enrichment)

**Context-Enriched Retrieval for RAG — Iris.ai**

This notebook produces the **baseline** retrieval scores on BEIR's **SCIDOCS** using the
`Octen/Octen-Embedding-0.6B` embedding model. *No enrichment* — plain query → embed → cosine
retrieval → BEIR metrics. These numbers are the reference that the later **HyDE** run is diffed against.

The embedding model is held **fixed** across every experiment; only the *representation* of
queries/chunks changes. We save the full ranked run + metrics + config so any later method can be
re-scored with identical code.

**Note on prompts:** the model defines a native asymmetric retrieval format — queries get an
`Instruct: ...` prefix, documents get a blank prompt. That is the model's intended usage (not
enrichment), so the baseline uses it and so will every later run.

## 1. Config & setup

In [10]:
import os, json, time, hashlib, random
from pathlib import Path
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import pytrec_eval

# ---- Config ---------------------------------------------------------------
MODEL_NAME = "Octen/Octen-Embedding-0.6B"
DATA_DIR   = Path("../scidocs")            # corpus.jsonl, queries.jsonl, qrels/test.tsv
SPLIT      = "test"                          # which qrels split to evaluate on
OUT_DIR    = Path("results/scidocs/baseline")
TOP_K      = 1000                            # ranking depth (>= max metric cutoff)
BATCH_SIZE = 32
SEED       = 42
RUN_TAG    = "baseline"

OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device     :", DEVICE)
print("data dir   :", DATA_DIR.resolve())
print("output dir :", OUT_DIR.resolve())

device     : mps
data dir   : /Users/katerina_dimitrova/Downloads/Iris.ai RAG research/nfcorpus
output dir : /Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/results/baseline


## 2. Load corpus, queries, and qrels

We evaluate only on the queries that appear in `qrels/<split>.tsv` (SCIDOCS test = 323 queries).

In [11]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_qrels(path):
    """TREC/BEIR qrels: tsv with header query-id\tcorpus-id\tscore."""
    qrels = {}
    with open(path, encoding="utf-8") as f:
        header = next(f)  # skip header line
        for line in f:
            qid, did, score = line.rstrip("\n").split("\t")
            score = int(score)
            if score > 0:                      # keep only judged-relevant
                qrels.setdefault(qid, {})[did] = score
    return qrels

# ---- Corpus: id -> (title, text) -----------------------------------------
corpus_rows = load_jsonl(DATA_DIR / "corpus.jsonl")
corpus = {r["_id"]: (r.get("title", "") or "", r.get("text", "") or "") for r in corpus_rows}
corpus_ids = list(corpus.keys())

# ---- Qrels for the chosen split ------------------------------------------
qrels = load_qrels(DATA_DIR / "qrels" / f"{SPLIT}.tsv")
eval_qids = set(qrels.keys())

# ---- Queries: keep only those in the eval split --------------------------
all_queries = {r["_id"]: r["text"] for r in load_jsonl(DATA_DIR / "queries.jsonl")}
queries = {qid: all_queries[qid] for qid in eval_qids if qid in all_queries}
query_ids = list(queries.keys())

print(f"corpus docs        : {len(corpus):>6}")
print(f"queries ({SPLIT})     : {len(queries):>6}  (of {len(all_queries)} total)")
print(f"judged query/doc   : {sum(len(v) for v in qrels.values()):>6} relevance pairs")
missing = eval_qids - set(all_queries)
if missing:
    print(f"WARNING: {len(missing)} qrels query-ids not found in queries.jsonl")

corpus docs        :   3633
queries (test)     :    323  (of 3237 total)
judged query/doc   :  12334 relevance pairs


## 3. Embed the corpus (cached)

Documents are embedded once with the model's `document` prompt and cached to disk. Every later
experiment reuses this exact corpus matrix — only the *query* side changes between baseline and HyDE.
Document text = `title + "\n" + text` (no enrichment).

In [12]:
# Reusable embedder (shared by baseline + every later method)
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 512   # SCIDOCS docs are short; cap for speed (full max is 32768)
print("max_seq_length:", model.max_seq_length, "| embedding dim:", model.get_sentence_embedding_dimension())

def embed(texts, prompt_name, desc):
    return model.encode(
        texts,
        prompt_name=prompt_name,        # 'query' or 'document' -> model's native prefixes
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,      # cosine == dot product
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 8137.35it/s]


max_seq_length: 512 | embedding dim: 1024


/var/folders/v1/ggly724d161gg06czzxjv4100000gn/T/ipykernel_4177/687104644.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("max_seq_length:", model.max_seq_length, "| embedding dim:", model.get_sentence_embedding_dimension())


In [13]:
CORPUS_EMB_PATH = OUT_DIR / "corpus_emb.npy"
CORPUS_IDS_PATH = OUT_DIR / "corpus_ids.json"

# Load cache iff it matches the current corpus id order; otherwise (re)embed.
cache_ok = False
if CORPUS_EMB_PATH.exists() and CORPUS_IDS_PATH.exists():
    cached_ids = json.loads(CORPUS_IDS_PATH.read_text())
    if cached_ids == corpus_ids:
        corpus_emb = np.load(CORPUS_EMB_PATH)
        cache_ok = True
        print("loaded cached corpus embeddings:", corpus_emb.shape)

if not cache_ok:
    doc_texts = [(t + "\n" + b).strip() for (t, b) in (corpus[i] for i in corpus_ids)]
    t0 = time.time()
    corpus_emb = embed(doc_texts, prompt_name="document", desc="corpus")
    np.save(CORPUS_EMB_PATH, corpus_emb)
    CORPUS_IDS_PATH.write_text(json.dumps(corpus_ids))
    print(f"embedded corpus {corpus_emb.shape} in {time.time()-t0:.1f}s")

loaded cached corpus embeddings: (3633, 1024)


## 4. Embed queries & retrieve (cosine, top-k)

In [14]:
t0 = time.time()
query_texts = [queries[q] for q in query_ids]
query_emb = embed(query_texts, prompt_name="query", desc="queries")
print(f"embedded {query_emb.shape[0]} queries in {time.time()-t0:.1f}s")

# Cosine similarity (both sides normalized) -> dot product. Top-k per query.
sims = query_emb @ corpus_emb.T                      # (Q, D)
k = min(TOP_K, sims.shape[1])
topk_idx = np.argpartition(-sims, k - 1, axis=1)[:, :k]

run = {}
for i, qid in enumerate(query_ids):
    idx = topk_idx[i]
    order = idx[np.argsort(-sims[i, idx])]           # sort the k candidates by score
    run[qid] = {corpus_ids[j]: float(sims[i, j]) for j in order}
print("retrieved top", k, "docs for", len(run), "queries")

Batches: 100%|██████████| 11/11 [00:02<00:00,  4.53it/s]

embedded 323 queries in 2.4s
retrieved top 1000 docs for 323 queries


## 5. Evaluate with BEIR-standard metrics (`pytrec_eval`)

In [15]:
# Cutoffs tuned for RAG: small-k matters most (only top-k chunks are fed to the LLM).
# success_k / Hit@k = 1 if >=1 relevant doc appears in the top k (RAG often needs just one good chunk).
measures = {"ndcg_cut.1,3,5,10", "recall.3,5,10,20,100", "map", "P.10", "recip_rank", "success.1,5,10"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels, measures)
per_query = evaluator.evaluate(run)

def avg(metric):
    return float(np.mean([per_query[q][metric] for q in per_query]))

metrics = {
    "NDCG@1":    avg("ndcg_cut_1"),
    "NDCG@3":    avg("ndcg_cut_3"),
    "NDCG@5":    avg("ndcg_cut_5"),
    "NDCG@10":   avg("ndcg_cut_10"),
    "Recall@3":  avg("recall_3"),
    "Recall@5":  avg("recall_5"),
    "Recall@10": avg("recall_10"),
    "Recall@20": avg("recall_20"),
    "Recall@100":avg("recall_100"),
    "Hit@1":     avg("success_1"),
    "Hit@5":     avg("success_5"),
    "Hit@10":    avg("success_10"),
    "MAP":       avg("map"),
    "P@10":      avg("P_10"),
    "MRR":       avg("recip_rank"),
}

# Save per-query NDCG@10 so the HyDE notebook can run win/loss + Wilcoxon vs this run.
(OUT_DIR / "per_query_ndcg10.json").write_text(
    json.dumps({q: per_query[q]["ndcg_cut_10"] for q in per_query}, indent=2))

HEADLINE = {"NDCG@10", "Recall@5", "Recall@10", "Hit@5"}
print(f"{'metric':<12} {'score':>8}")
print("-" * 26)
for m, v in metrics.items():
    star = "  <-- headline" if m in HEADLINE else ""
    print(f"{m:<12} {v:>8.4f}{star}")

metric          score
--------------------------
NDCG@1         0.4737
NDCG@3         0.4206
NDCG@5         0.4002
NDCG@10        0.3654  <-- headline
Recall@3       0.1112
Recall@5       0.1402  <-- headline
Recall@10      0.1747  <-- headline
Recall@20      0.2167
Recall@100     0.3424
Hit@1          0.4923
Hit@5          0.6842  <-- headline
Hit@10         0.7245
MAP            0.1911
P@10           0.2687
MRR            0.5851


## 6. Save outputs (for diffing against HyDE later)

- `run.tsv` — full ranked run in TREC format (re-scoreable by any later method)
- `metrics.json` — all metric values
- `config.json` — every parameter needed to reproduce this run
- `summary.md` — human-readable headline table

In [16]:
# 1) TREC run file: query-id Q0 corpus-id rank score tag
with open(OUT_DIR / "run.tsv", "w") as f:
    for qid in query_ids:
        for rank, (did, score) in enumerate(run[qid].items(), start=1):
            f.write(f"{qid}\tQ0\t{did}\t{rank}\t{score:.6f}\t{RUN_TAG}\n")

# 2) metrics
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

# 3) config / provenance
def file_sha1(p):
    h = hashlib.sha1()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

config = {
    "run_tag": RUN_TAG,
    "enrichment": "none (baseline)",
    "model_name": MODEL_NAME,
    "embedding_dim": int(corpus_emb.shape[1]),
    "pooling": "last-token",
    "normalized": True,
    "query_prompt": model.prompts.get("query"),
    "document_prompt": model.prompts.get("document"),
    "max_seq_length": model.max_seq_length,
    "device": DEVICE,
    "dataset": "SCIDOCS",
    "split": SPLIT,
    "n_corpus": len(corpus),
    "n_queries": len(queries),
    "top_k": k,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "corpus_sha1": file_sha1(DATA_DIR / "corpus.jsonl"),
    "qrels_sha1": file_sha1(DATA_DIR / "qrels" / f"{SPLIT}.tsv"),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

# 4) summary.md
lines = [
    "# SCIDOCS Baseline — Summary", "",
    f"- Model: `{MODEL_NAME}`",
    f"- Dataset: SCIDOCS ({SPLIT}) — {len(corpus)} docs, {len(queries)} queries",
    f"- Enrichment: none (baseline)",
    f"- Date: {config['timestamp']}", "",
    "| Metric | Score |", "|---|---|",
]
lines += [f"| {m} | {v:.4f} |" for m, v in metrics.items()]
(OUT_DIR / "summary.md").write_text("\n".join(lines) + "\n")

print("saved to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)

saved to /Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/results/baseline
   config.json
   corpus_emb.npy
   corpus_ids.json
   metrics.json
   per_query_ndcg10.json
   run.tsv
   summary.md
